## 🛠️ Initialize

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import os
import re
import requests
from collections import Counter, defaultdict
from sklearn.ensemble import RandomForestClassifier
from datetime import timedelta, datetime
import random
import joblib
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\ProgramData\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\ProgramData\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
def degrees_to_direction(deg):

    sectors = [
        (22.5, "N"),
        (67.5, "NE"),
        (112.5, "E"),
        (157.5, "SE"),
        (202.5, "S"),
        (247.5, "SO"),
        (292.5, "O"),
        (337.5, "NO"),
        (360, "N")
    ]

    for limit, direction in sectors:
        if deg < limit:
            return direction


weather_number = 0

def get_daily_weather(date):
    global weather_number

    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={36.4621}"
        f"&longitude={7.4261}"
        f"&start_date={date}"
        f"&end_date={date}"
        "&hourly=temperature_2m,wind_speed_10m,wind_direction_10m"
        "&timezone=Europe%2FLondon"
    )

    response = requests.get(url)
    data = response.json()
    #print(data)
    hourly = data["hourly"]

    temperatures = hourly["temperature_2m"]
    wind_speeds = hourly["wind_speed_10m"]
    wind_directions = hourly["wind_direction_10m"]

    # averages
    avg_temp = round(sum(temperatures) / len(temperatures), 2)
    avg_wind_speed = round(sum(wind_speeds) / len(wind_speeds), 2)

    # convert hourly directions
    cardinal_dirs = [
        degrees_to_direction(d)
        for d in wind_directions
    ]

    counts = Counter(cardinal_dirs)

    # sorted by frequency
    top_dirs = counts.most_common()

    # if only one dominant direction exists
    if len(top_dirs) == 1:
        final_direction = top_dirs[0][0]

    else:
        first_dir, first_count = top_dirs[0]
        second_dir, second_count = top_dirs[1]

        # only include second direction if meaningful
        if second_count >= first_count * 0.7:
            final_direction = f"{first_dir}+{second_dir}"
        else:
            final_direction = first_dir

    weather = {
    "temperature": float(avg_temp),
    "wind_speed": float(avg_wind_speed),
    "wind_direction": str(final_direction)
    }

    weather_number += 1

    print(f'weather for {date}: {weather}')
    print (f'weather_n : {weather_number}')
    return weather


def get_season(month):
    if month in [3, 4, 5]:
        return 'SPRING'
    elif month in [6, 7, 8]:
        return 'SUMMER'
    elif month in [9, 10, 11]:
        return 'AUTUMN'
    else:
        return 'WINTER'
    

def compute_fire_percentages(df):
    total = df["SURF_TOTAL"].sum()

    if pd.isna(total) or total == 0:
        return pd.Series({
            "FORET_PERCENT": 0,
            "MAQUIS_PERCENT": 0,
            "BROUSSAILLES_PERCENT": 0
        })

    return pd.Series({
        "FORET_PERCENT": (df["TOT_FORET"].sum() / total) * 100,
        "MAQUIS_PERCENT": (df["TOT_MAQUIS"].sum() / total) * 100,
        "BROUSSAILLES_PERCENT": (df["TOT_BROUSSAILLES"].sum() / total) * 100
    })

In [ ]:
# Charger le fichier
df_raw = pd.read_excel("raw_dataset.xlsx", header=None)

print(df_raw.shape)
df_raw.head(5)

In [ ]:
df1 = df_raw.iloc[11:].reset_index(drop=True)
df1.columns = [
    "ID", "DAIRA", "COMMUNE", "FORET",
    "LONGITUDE", "LATITUDE", "CART_NAME",
    
    "DATE_DECL", "HEURE_DECL",
    "DATE_INT", "HEURE_INT",
    "DATE_EXT", "HEURE_EXT",
    
    "ESSENCE",
    
    "DOM_FORET", "DOM_MAQUIS", "DOM_BROUSSAILLES", "DOM_ALFA", "DOM_AUTRES",
    "PRIV_FORET", "PRIV_MAQUIS", "PRIV_BROUSSAILLES", "PRIV_ALFA", "PRIV_AUTRES",
    
    "TOT_FORET", "TOT_MAQUIS", "TOT_BROUSSAILLES", "TOT_ALFA", "TOT_AUTRES",
    "SURF_TOTAL",
    
    "CAUSE", "AUTEUR", "SIGNALE",
    "METEO", "ORGANISMES",
    "DEGATS", 'ENQUETE'
]

df1 = df1.dropna(how="all").reset_index(drop=True)
df1

## 🧹 Clean

In [ ]:
df2 = df1.replace(["", " ", "NaN", "nan", "NULL", "null", "None", "-"], np.nan)

#strip whitespace from string columns
str_cols = df2.select_dtypes(include=['object']).columns

df2[str_cols] = df2[str_cols].apply(
    lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x)
)

nan_percent = (df2.isna().mean() * 100).sort_values(ascending=False)
print(nan_percent)

In [ ]:
#drop columns with more than 90% missing values + duplicated columns + uncontributing columns
df3  = df2.dropna(thresh=len(df2)*0.1, axis=1)
df3 = df3.drop(columns=['ID', 'DOM_FORET', 'DOM_MAQUIS', 'DOM_BROUSSAILLES', 'ENQUETE', 'AUTEUR'])

In [ ]:
df4 = df3.copy()
# make TOT cols missing to 0
tot_cols = ['TOT_FORET', 'TOT_MAQUIS', 'TOT_BROUSSAILLES']
for col in tot_cols:
    df4[col] = df4[col].fillna(0)

In [ ]:
df5 = df4.copy()

# Convert to datetime first
date_cols = ['DATE_DECL', 'DATE_INT', 'DATE_EXT']
time_cols = ['HEURE_DECL', 'HEURE_INT', 'HEURE_EXT']

for col in date_cols:
    df5[col] = pd.to_datetime(df5[col], errors='coerce')

for col in time_cols:
    df5[col] = pd.to_datetime(df5[col], format='%H:%M:%S', errors='coerce')

# Date features
df5['DECL_YEAR']  = df5['DATE_DECL'].dt.year
df5['DECL_MONTH'] = df5['DATE_DECL'].dt.month

df5['INT_YEAR']   = df5['DATE_INT'].dt.year
df5['INT_MONTH']  = df5['DATE_INT'].dt.month
df5['INT_DAY']    = df5['DATE_INT'].dt.day

df5['EXT_YEAR']   = df5['DATE_EXT'].dt.year
df5['EXT_MONTH']  = df5['DATE_EXT'].dt.month
df5['EXT_DAY']    = df5['DATE_EXT'].dt.day

# Time features
df5['DECL_HOUR'] = df5['HEURE_DECL'].dt.hour
df5['INT_HOUR']  = df5['HEURE_INT'].dt.hour
df5['EXT_HOUR']  = df5['HEURE_EXT'].dt.hour

In [ ]:
# clean categorical features
df6 = df5.copy()


df6['DAIRA'] = df6['DAIRA'].replace('KHEZARAS', 'KHEZARA')
df6['DAIRA'] = df6['DAIRA'].replace("HAMMAM N'BAILS", "HAMMAM N'BAIL")

In [ ]:
df6['FORET'] = df6['FORET'].str.replace(r'^KAF\s+EL\s+', 'KAF ', regex=True)
#replace KAF with KEF

df6['FORET'] = df6['FORET'].str.replace(r'^KAF\s+', 'KEF ', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'^KEF\s+EL\s+', 'KEF ', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'^KEF\s+', 'KEF ', regex=True)

df6['FORET'] = df6['FORET'].str.replace(r'\b(\w+)\s+\1\b', r'\1', regex=True)

df6['FORET'] = df6['FORET'].str.replace(r'\s+', ' ', regex=True)
df6['FORET'] = df6['FORET'].str.strip()


## change DJEBEL & DJEBL & DJBEL with DJ
df6['FORET'] = df6['FORET'].str.replace(r'\bDJEBEL\b', 'DJ', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'\bDJEBL\b', 'DJ', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'\DJBEL\b', 'DJ', regex=True)


In [ ]:

forest_standardization = [['AIN GUESSAB', 'AIN GUESBA', 'AIN LEGSAB'],
 ['OUED EL MENDJEL', 'OUED MENDJEL'],
 ['AIN EL KARMA', 'AIN KERMA', 'EL KARMA'],
 ['EL MADJENE', 'EL MAJEN', 'AIN EL MADJENE'],
 ['BENI SALAH (AIN ZANA)',
  'FORET BENI SALH',
  'F D BENI SALAH (LOURIDA)',
  'EL REHAB (Béni Salah)',
  'BENI SALAH SÉRIE 12',
  'BENI SALAH SÉRIE 15', 'BENI SALAH'],
 ['DJ BOUGHERISSINE', 'BOUGHERISSINE', 'BOUGHRESINE'],
 ['DJ MARMOURA', 'MERMOURA'],
 ['DJ LAZREG', 'DJ LEZREG'],
 ['BERAIMA', 'BERIMA'],
 ['BOUROUISSINE', 'BOUROUISNE'],
 ['DJ ASKAR', 'DJ EL ASKAR', 'DJ LASKRE', 'DJ ASEKRE'],
 ['DJ MEKHALFA', 'DJ MEKHALHA', 'EL MEKHALFA'],
 ['DJ TAYA', 'DJ TAYA (CHOUIHA)', 'TAYA', 'TAYA BENI AHMED'],
 ['FEROUDJA', 'DJ FEROUDJA', 'OUED GHANEM FAROUJA'],
 ['OUE GHANEM', 'OUED GHANEM'],
 ['KEF ALI ARIANE', 'KEF SIDI ALI EL ARIANE', 'SIDI GUEMAM', 'SIDI ALI EL AARIAN'],
 ['KEF SAREK', 'KEF SERAK', 'KEF SERRAK', 'KER SERAK'],
 ['KEF LAHDJAR', 'HADJAR EL MENCHAR'],
 ['KEF KORATH', 'KEF KOURATH'],
 ['KEF RAMOUL', 'KEF RAMOUL (LOURIDA)', 'CHt EL RAMOUL', 'CHAABET RAMOUL'],
 ['EL MAZA', 'EL MAZA SEGHIRA'],
 ['EL MRIHL', 'EL MERIDJ'],
 ['EL MENCHAR', 'EL MUNCHAR', 'EL MUNCHORA', 'MENCHAR'],
 ['LOURAIDA ESSGHIRA', 'LOUREIDA', 'LOURIDA'],
 ['EL GLAA', 'LEGLAA', 'LEGULALA', 'GUELALA', 'GELALA'],
 ['OLED GUERARA', 'OULED GHERARA', 'OULED GUERARA', 'OULED GUERARA MANCHOURA', 'MECHTAT GUERARA'],
 ['SEBAA MEZAIR', 'SEBAA MEZIAR', 'SEBAA MZAZET'],
 ['SIDI REZIGU', 'SIDI REZIK'],
 ['SIDI NASR', 'SIDI NASSER'],
 ['TOUIFEZA', 'TOUIFZA', 'MECHTAT TOUIFEZA'],
 ['ZIZOUA', 'ZGHAOULA', 'ZÉZOUA'],
 ['EL HIMEUR', 'HIMEUR', 'HOUZ EL HIMRE'],
 ['HAMMAM OULED ALI', 'HMAM OULED ALI'],
 ['MECHTAT AIN RIHANA', 'RIHANA', 'AIN RIHANA'],
 ['MECHTAT TEBASSA', 'TEBASSA'],
 ['MECHTAT SILA', 'MT SILA', 'SILAT', 'SILA'],
 ['OUED EL MEJNOUNA', 'OUED EL MEDJNOUNA'],
 ['FAROUDJA', 'FEROUDJA', 'FAROUJA', 'FERODJA', 'FROUJA'],
 ['KAF SIDI ALI LAARIANE', 'KEF S/ALI EL ARIANE', 'KEF SIDI ALI ARIANE', 'KEF SIDI ALI EL ARIANE'],
 ['DJ BOUGHERISSINE', 'DJ BOUGHRESINE', 'DJ BOUGHRESSINE', 'DJ BOUGRÉNISSA'],
 ['EL MERAJIL', 'EL MERIHIL', 'LEMRIHIL', 'EL MRIHL'],
 ['EL MAZA ESSGHIRA', 'EL MAZA SAGHIRA', 'EL MAZA SEGHIRA'],
 ['KEF SERRAK', 'KEF SERAK', 'KER SERAK'],
 ['DJ MERMOURA', 'DJ MARMOURA'],
 ['F D SEFAHLI', "S'FAHLI", 'SFAHLI'],
 ['GHENAJI', 'GHENADJI'],
 ['GUFAZA', 'GUEFAZA'],
 ['FEDJ CHEAIR', 'FEDJ CHAIR'],
 ['KEF SIDI ALI LAARIANE', 'KEF SIDI ALI EL ARIANE'],
 ['SAFIAT SAID', 'SAFIET SAID'],
 ['F D MAHOUNA', 'MAHOUNA'],
 ['MECHTAT AIN RIHANA', 'AIN RIHANA'],
 ['MECHTAT METOUIA', 'METOUIA'],
 ['MECHTAT LAKHOUARNA', 'LEKHOUARNA'],
 ['MECHTAT EL BAGRAT', 'AIN EL BAGRAT'],
 ['BENI AHMED KEF SERRAK', 'KEF SERRAK'],
 ['DJ EL BATTOUM', 'EL BATTOUM']]

# 1. تحويل قائمة القوائم إلى قاموس
cleaning_dict = {}
for group in forest_standardization:
    target_word = group[-1]  # الكلمة الأخيرة
    for word in group[:-1]:  # كل الكلمات ما عدا الأخيرة
        cleaning_dict[word] = target_word

df6['FORET'] = df6['FORET'].replace(cleaning_dict)



In [ ]:
# Get unique FORET values, drop NaN, sort alphabetically
base_path = "tests"
forets = sorted(df6['FORET'].dropna().unique())

# Create a new dataframe
foret_df = pd.DataFrame({'FORET': forets})

# Save to Excel file
foret_df.to_csv(f"{base_path}/forests_alphabetical.csv", index=False)

print("CSV file created: forests_alphabetical.csv")

In [ ]:
# normalize SIGNALE
df6['SIGNALE'] = (
    df6['SIGNALE']
    .astype(str)
    .str.upper()
    .str.replace(r'[/,\-]', '+', regex=True)   # replace / , -
    .str.replace(r'\s*\+\s*', '+', regex=True) # clean spaces around +
    .str.replace(r'\s+', '+', regex=True)      # remaining spaces -> +
    .str.replace(r'\++', '+', regex=True)      # remove duplicate +
    .str.strip('+ ')
)

# normalize ORGANISMES
df6['ORGANISMES'] = (
    df6['ORGANISMES']
    .astype(str)
    .str.upper()
    .str.replace(r'[/,\-]', '+', regex=True)
    .str.replace(r'\s*\+\s*', '+', regex=True)
    .str.replace(r'\s+', '+', regex=True)
    .str.replace(r'\++', '+', regex=True)
    .str.strip('+ ')
)

# -------------------------
# 1. basic normalization
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\s+', ' ', regex=True).str.strip()

# -------------------------
# 2. unify separators
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\s*\+\s*', '+', regex=True)
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\s*-\s*', '+', regex=True)



# -------------------------
# 3. standardize labels
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace("CL (DEGR)", "CL", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("BROUSS", "BRS", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("BROSS", "BRS", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("CYPRE", "CYP", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("MAQUIS", "MAQ", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("T.PARCOUR", "TPARCOUR", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("تشجير", "REB", regex=False)

# -------------------------
# 4. clean bad separators
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace(' ', '+')  # replace space with +
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\++', '+', regex=True)   # remove multiple +
df6['ESSENCE'] = df6['ESSENCE'].str.strip('+ ')                        # trim edge +

# extract (REB) or any parenthesis content
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\(([^)]+)\)', r'+\1', regex=True)

# clean multiple +
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\++', '+', regex=True)

# clean edges
df6['ESSENCE'] = df6['ESSENCE'].str.strip('+ ')

# -------------------------
# 5. final cleanup
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].replace(['', ' ', 'nan'], pd.NA)
df6['ESSENCE'] = df6['ESSENCE'].fillna(df6['ESSENCE'].mode()[0])  # fill missing with mode

In [ ]:
df7 = df6.copy()
df7[['METEO_TEMP', 'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION']] = \
df7['METEO'].str.split(' - ', expand=True)

# 2. clean temperature (remove ° and convert to float)
df7['METEO_TEMP'] = (
    df7['METEO_TEMP']
    .str.replace('°', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 3. clean wind speed (remove Km/h and convert to float)
df7['METEO_VENTE_VITESSE'] = (
    df7['METEO_VENTE_VITESSE']
    .str.replace('Km/h', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 4. clean direction
df7['METEO_VENTE_DIRECTION'] = (
    df7['METEO_VENTE_DIRECTION']
    .str.upper()
    .str.strip()
)
# 5. replace the direction full word with symbol
direction_map = {
    'NORD': 'N',
    'SUD': 'S',
    'EST': 'E',
    'OUEST': 'O',
    'NORD-EST': 'NE',
    'NORD-OUEST': 'NO',
    'SUD-EST': 'SE',
    'SUD-OUEST': 'SO'
}

df7['METEO_VENTE_DIRECTION'] = df7['METEO_VENTE_DIRECTION'].replace(direction_map)
df7['METEO_VENTE_DIRECTION'] = df7['METEO_VENTE_DIRECTION'].fillna('').str.replace('/', '+')

df7["METEO_TEMP"] = pd.to_numeric(df7["METEO_TEMP"], errors="coerce")
df7["METEO_VENTE_VITESSE"] = pd.to_numeric(df7["METEO_VENTE_VITESSE"], errors="coerce")
df7["METEO_VENTE_DIRECTION"] = df7["METEO_VENTE_DIRECTION"].astype("string")

df7.drop(columns=['METEO'], inplace=True)
df7.head()

In [ ]:
df8 = df7.copy()

# Cache to avoid repeated API calls
weather_cache = {}

# 🤖 Fill missing Weather with API
for idx, row in df8.iterrows():

    missing_weather = (
        pd.isna(row["METEO_TEMP"]) or
        pd.isna(row["METEO_VENTE_VITESSE"]) or
        pd.isna(row["METEO_VENTE_DIRECTION"]) or
        row["METEO_VENTE_DIRECTION"] == ""
    )

    if missing_weather:
        try:
            date = pd.to_datetime(row["DATE_DECL"]).strftime("%Y-%m-%d")

            # 💡 cache API results
            if date not in weather_cache:
                weather_cache[date] = get_daily_weather(date)

            weather = weather_cache[date]

            temp = weather["temperature"]
            wind_speed = weather["wind_speed"]
            wind_direction = weather["wind_direction"]

            # 💡 safe assignment
            df8.loc[idx, "METEO_TEMP"] = float(temp)
            df8.loc[idx, "METEO_VENTE_VITESSE"] = float(wind_speed)
            df8.loc[idx, "METEO_VENTE_DIRECTION"] = str(wind_direction)

        except Exception as e:
            print(f"Error at row {idx}: {e} - weather : {weather_cache[date]}")

In [ ]:
# Lang and coordinates cleaning
df9 = df8.copy()
def extract_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    # replace comma decimal → dot
    x = x.replace(',', '.')

    # remove spaces
    x = x.strip()

    # remove ranges → take first value
    if '-' in x:
        x = x.split('-')[0]

    # extract first number found
    match = re.search(r'[-+]?\d*\.?\d+', x)
    if match:
        return float(match.group())

    return np.nan

def dms_to_dd(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    pattern = r'(\d+)[°\s]+(\d+)[\'\s]+([\d\.]+)"?\s*([NSEW])?'
    m = re.search(pattern, x)

    if not m:
        return np.nan

    deg, minutes, seconds, direction = m.groups()

    dd = float(deg) + float(minutes)/60 + float(seconds)/3600

    if direction in ['S', 'W']:
        dd = -dd

    return dd

def clean_coord(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    # case 1: DMS format
    if "°" in x:
        return dms_to_dd(x)

    # case 2: numeric / UTM / range
    return extract_number(x)

df9['LONGITUDE'] = df9['LONGITUDE'].apply(clean_coord)
df9['LATITUDE']  = df9['LATITUDE'].apply(clean_coord)

df9['LOCATION_NAME'] = (
    df9['CART_NAME']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df9.drop(columns=['CART_NAME'], inplace=True)


In [ ]:
#save the cleaned dataset
df9.to_excel('clean_dataset.xlsx', index=False)

## ⚠️THEME 1 : DANGER AREAS PREDECTION

### 👩🏻‍🍳 Prepare Data

In [ ]:
# remove unnecessary columns
df10 = pd.read_excel('clean_dataset.xlsx')
unc_cols = ['CAUSE', 'SIGNALE', 'ORGANISMES', 'DEGATS',
'INT_YEAR', 'INT_MONTH', 'INT_DAY', 'EXT_YEAR', 'EXT_MONTH', 'EXT_DAY',
'INT_HOUR', 'EXT_HOUR', 'DECL_HOUR', 'DATE_INT', 'HEURE_INT', 'DATE_EXT', 'HEURE_EXT',
'HEURE_DECL', 'LONGITUDE', 'LATITUDE', 'LOCATION_NAME', 'DECL_DAY']
df10 = df10.drop(columns=unc_cols)
#rename decl-times to just times
df10 = df10.rename(columns={'DECL_YEAR': 'YEAR', 'DECL_MONTH': 'MONTH', 'DATE_DECL': 'DATE'})
df10.columns

In [ ]:
df11 = df10.copy()
df11['WEEK'] = df11['DATE'].dt.isocalendar().week
df11['SEASON'] = df11['MONTH'].apply(get_season)
df11.head()

### ✂️ Split Tain/Test

In [ ]:
folder = "Cleaned Data/Theme1"
#  make sure df is sorted by date first
df11 = df11.sort_values("DATE").reset_index(drop=True)

#save full cleaned dataset
df11.to_excel(f"{folder}/clean_dataset.xlsx", index=False)

# compute split index (80% train, 20% test)
split_idx = int(len(df11) * 0.8)

# split
df_train = df11.iloc[:split_idx].copy()
df_test = df11.iloc[split_idx:].copy()

# (optional) sanity check
print("Total:", len(df11))
print("Train:", len(df_train))
print("Test:", len(df_test))

# save to Excel
# folder path

os.makedirs(folder, exist_ok=True)

# save files
df_train.to_excel(f"{folder}/split_train.xlsx", index=False)
df_test.to_excel(f"{folder}/split_test.xlsx", index=False)

print("Saved train and test files in:", folder)

In [ ]:
#for train
df12 = pd.read_excel("Cleaned Data/Theme1/split_train.xlsx")

In [ ]:
#for test
df12 = pd.read_excel("Cleaned Data/Theme1/split_test.xlsx")

In [ ]:
df12['FORET'].value_counts().value_counts().sort_index()

### 🧊 Freeze Values (train)

In [ ]:
# 1. fire count per DAIRA
daira_counts = df12['DAIRA'].value_counts().reset_index()
daira_counts.columns = ['DAIRA', 'FIRE_COUNT']

# 2. define groups (TRAIN-based logic)
strong_dairas = daira_counts[daira_counts['FIRE_COUNT'] >= 80]['DAIRA'].tolist()

medium_dairas = daira_counts[
    (daira_counts['FIRE_COUNT'] < 80) &
    (daira_counts['FIRE_COUNT'] >= 20)
]['DAIRA'].tolist()

weak_dairas = daira_counts[daira_counts['FIRE_COUNT'] < 20]['DAIRA'].tolist()

In [ ]:
commune_counts = df12['COMMUNE'].value_counts().reset_index()
commune_counts.columns = ['COMMUNE', 'FIRE_COUNT']

strong_communes = commune_counts[commune_counts['FIRE_COUNT'] >= 20]['COMMUNE'].tolist()

medium_communes = commune_counts[
    (commune_counts['FIRE_COUNT'] < 20) &
    (commune_counts['FIRE_COUNT'] >= 4)
]['COMMUNE'].tolist()

weak_communes = commune_counts[commune_counts['FIRE_COUNT'] < 4]['COMMUNE'].tolist()

In [ ]:
forest_counts = df12['FORET'].value_counts().reset_index()
forest_counts.columns = ['FORET', 'FIRE_COUNT']

strong_forests = forest_counts[forest_counts['FIRE_COUNT'] >= 6]['FORET'].tolist()

medium_forests = forest_counts[
    (forest_counts['FIRE_COUNT'] < 6) &
    (forest_counts['FIRE_COUNT'] >= 3)
]['FORET'].tolist()

weak_forests = forest_counts[forest_counts['FIRE_COUNT'] < 3]['FORET'].tolist()

In [ ]:
const_wind_directions = ['N', 'NE', 'E', 'SE', 'S', 'SO', 'O', 'NO']
const_essences = [
 'BRS',
 'CL',
 'CV',
 'CYP',
 'CZ',
 'EUC',
 'ESSENCE',
 'FORETS',
 'GUENDOUL',
 'MAQ',
 'PA',
 'REB',
 'TPARCOUR'
]
const_seasons = ['SUMMER', 'AUTUMN', 'WINTER', 'SPRING']

### 👷 Feature Engennering

In [ ]:
df13 = df12.copy()

# =========================
# 1. DAIRA
# =========================

df13['DAIRA_FIRE_COUNT'] = (
    df13['DAIRA']
    .map(daira_counts.set_index('DAIRA')['FIRE_COUNT'])
    .fillna(0)
    .astype(int)
)

def map_daira(x):
    if x in strong_dairas:
        return x
    else:
        return 'OTHER'

df13['DAIRA_GROUP'] = df13['DAIRA'].apply(
    lambda x: 'STRONG' if x in strong_dairas else
              'MEDIUM' if x in medium_dairas else
              'WEAK'   if x in weak_dairas else 'OTHER'
)

df13['DAIRA_2'] = df13['DAIRA'].apply(map_daira)


# =========================
# 2. COMMUNE
# =========================

df13['COMMUNE_FIRE_COUNT'] = (
    df13['COMMUNE']
    .map(commune_counts.set_index('COMMUNE')['FIRE_COUNT'])
    .fillna(0)
    .astype(int)
)

def map_commune(x):
    if x in strong_communes:
        return x
    else:
        return 'OTHER'

df13['COMMUNE_GROUP'] = df13['COMMUNE'].apply(
    lambda x: 'STRONG' if x in strong_communes else
              'MEDIUM' if x in medium_communes else
              'WEAK'   if x in weak_communes else 'OTHER'
)

df13['COMMUNE_2'] = df13['COMMUNE'].apply(map_commune)


# =========================
# 3. FORET
# =========================

df13['FORET_FIRE_COUNT'] = (
    df13['FORET']
    .map(forest_counts.set_index('FORET')['FIRE_COUNT'])
    .fillna(0)
    .astype(int)
)

def map_forest(x):
    if x in strong_forests:
        return x
    else:
        return 'OTHER'

df13['FORET_GROUP'] = df13['FORET'].apply(
    lambda x: 'STRONG' if x in strong_forests else
              'MEDIUM' if x in medium_forests else
              'WEAK'   if x in weak_forests else 'OTHER'
)

df13['FORET_2'] = df13['FORET'].apply(map_forest)
df13.head()

### 🧊 Freeze Forest Context (train)

In [ ]:
forest_context = (
    df13.groupby("FORET")
    .agg({
        "DAIRA": "first",
        "COMMUNE": "first",
        "TOT_FORET": "mean",
        "TOT_MAQUIS": "mean",
        "TOT_BROUSSAILLES": "mean",
        "SURF_TOTAL": "mean",
        "ESSENCE": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'FORET_FIRE_COUNT': 'mean',
        'DAIRA_FIRE_COUNT': 'mean',
        'COMMUNE_FIRE_COUNT': 'mean',
        "FORET_GROUP": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'DAIRA_GROUP': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'COMMUNE_GROUP': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "FORET_2": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "DAIRA_2": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "COMMUNE_2": lambda x: x.mode().iloc[0] if not x.mode().empty else None

    })
    .reset_index()
)

g = df13.groupby("FORET")

pct_features = pd.DataFrame({
    "FORET": g.size().index,
    "FORET_PERCENT": (g["TOT_FORET"].sum() / g["SURF_TOTAL"].sum()) * 100,
    "MAQUIS_PERCENT": (g["TOT_MAQUIS"].sum() / g["SURF_TOTAL"].sum()) * 100,
    "BROUSSAILLES_PERCENT": (g["TOT_BROUSSAILLES"].sum() / g["SURF_TOTAL"].sum()) * 100,
}).reset_index(drop=True)

forest_context = forest_context.merge(pct_features, on="FORET", how="left")

### 🧊 Append Forest Context (test)

In [ ]:
# -------------------------
# 1. Identify missing FORETs
# -------------------------
existing_forests = set(forest_context["FORET"])
missing_forests = set(df13["FORET"]) - existing_forests


# -------------------------
# 2. Build aggregated context ONLY for missing forests
# -------------------------
new_context_rows = (
    df13[df13["FORET"].isin(missing_forests)]
    .groupby("FORET")
    .agg({
        "DAIRA": "first",
        "COMMUNE": "first",

        "TOT_FORET": "mean",
        "TOT_MAQUIS": "mean",
        "TOT_BROUSSAILLES": "mean",
        "SURF_TOTAL": "mean",

        "ESSENCE": lambda x: x.mode().iloc[0] if not x.mode().empty else None,

        "FORET_FIRE_COUNT": "mean",
        "DAIRA_FIRE_COUNT": "mean",
        "COMMUNE_FIRE_COUNT": "mean",

        "FORET_GROUP": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "DAIRA_GROUP": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "COMMUNE_GROUP": lambda x: x.mode().iloc[0] if not x.mode().empty else None,

        "FORET_2": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "DAIRA_2": lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        "COMMUNE_2": lambda x: x.mode().iloc[0] if not x.mode().empty else None
    })
    .reset_index()
)

g = df13.groupby("FORET")

pct_features = pd.DataFrame({
    "FORET": g.size().index,
    "FORET_PERCENT": (g["TOT_FORET"].sum() / g["SURF_TOTAL"].sum()) * 100,
    "MAQUIS_PERCENT": (g["TOT_MAQUIS"].sum() / g["SURF_TOTAL"].sum()) * 100,
    "BROUSSAILLES_PERCENT": (g["TOT_BROUSSAILLES"].sum() / g["SURF_TOTAL"].sum()) * 100,
}).reset_index(drop=True)

new_context_rows = new_context_rows.merge(pct_features, on="FORET", how="left")


forest_context = pd.concat(
    [forest_context, new_context_rows],
    ignore_index=True
)

forest_context = forest_context.drop_duplicates(subset=["FORET"]).reset_index(drop=True)

print("Updated forest_context size:", len(forest_context))

### 🌲 Apply Forrest Context

In [ ]:
df14 = df13.merge(forest_context, on="FORET", how="left", suffixes=("", "_ctx"))

for col in forest_context.columns:
    if col != "FORET" and col in df13.columns:
        df14[col] = df14[col + "_ctx"]
        df14.drop(columns=[col + "_ctx"], inplace=True)


## 🏭 0 Label Generation

In [ ]:
df15 = df14.copy()

In [ ]:
# variables
def build_fire_set(df_fire):
    return set(zip(df_fire['FORET'], df_fire['DATE']))

def init_shift_stats():
    return {
        "negative": 0,
        "positive": 0,
        "short": 0,
        "long": 0
    }

forest_pool = df15['FORET'].unique().tolist()
weather_cache = {}

MIN_DATE = datetime(1940, 1, 1)
MAX_DATE = datetime(2026, 5, 2)

seen_samples = set()

forest_usage = defaultdict(int)
pair_usage = defaultdict(int)

#diffrent cetween test and train
foret_base_rep=3



In [ ]:
def get_cached_weather(date):
    
    key = date.strftime("%Y-%m-%d")
    
    if key in weather_cache:
        return weather_cache[key]
    
    weather = get_daily_weather(key)
    weather_cache[key] = weather
    
    return weather

In [ ]:
#change row helpers
def change_row(row, date):

    # 1. DATE FEATURES
    year = date.year
    month = date.month
    
    week = date.isocalendar().week
    
    season = get_season(month)

    # 2. WEATHER FROM API
    weather = get_cached_weather(date)

    # 3. BUILD UPDATED ROW
    new_row = row.copy()
    
    new_row['DATE'] = date
    new_row['YEAR'] = year
    new_row['MONTH'] = month
    new_row['WEEK'] = week
    new_row['SEASON'] = season
    
    new_row['METEO_TEMP'] = weather['temperature']
    new_row['METEO_VENTE_VITESSE'] = weather['wind_speed']
    new_row['METEO_VENTE_DIRECTION'] = weather['wind_direction']
    
    return new_row


def change_row_forest(row, new_forest):
    
    ctx = forest_context[forest_context['FORET'] == new_forest].iloc[0]
    
    new_row = row.copy()

    # FOREST SWAP ONLY
    new_row['FORET'] = new_forest
    new_row['DAIRA'] = ctx['DAIRA']
    new_row['COMMUNE'] = ctx['COMMUNE']
    new_row['FORET_2'] = ctx['FORET_2']
    new_row['DAIRA_2'] = ctx['DAIRA_2']
    new_row['COMMUNE_2'] = ctx['COMMUNE_2']
    new_row['FORET_FIRE_COUNT'] = ctx['FORET_FIRE_COUNT']
    new_row['DAIRA_FIRE_COUNT'] = ctx['DAIRA_FIRE_COUNT']
    new_row['COMMUNE_FIRE_COUNT'] = ctx['COMMUNE_FIRE_COUNT']
    new_row['FORET_GROUP'] = ctx['FORET_GROUP']
    new_row['DAIRA_GROUP'] = ctx['DAIRA_GROUP']
    new_row['COMMUNE_GROUP'] = ctx['COMMUNE_GROUP']
    new_row['ESSENCE'] = ctx['ESSENCE']
    new_row['FORET_PERCENT'] = ctx['FORET_PERCENT']
    new_row['MAQUIS_PERCENT'] = ctx['MAQUIS_PERCENT']
    new_row['BROUSSAILLES_PERCENT'] = ctx['BROUSSAILLES_PERCENT']
    new_row['TOT_FORET'] = ctx['TOT_FORET']
    new_row['TOT_MAQUIS'] = ctx['TOT_MAQUIS']
    new_row['TOT_BROUSSAILLES'] = ctx['TOT_BROUSSAILLES']
    new_row['SURF_TOTAL'] = ctx['SURF_TOTAL']

    return new_row

In [ ]:
#choosing helpers

def shift_date(base_date, days_shift):
    return base_date + timedelta(days=days_shift)


def random_shift_date(base_date):

    direction = random.choice([-1, 1])
    days_shift = random.randint(7, 358) * direction

    candidate = base_date + timedelta(days=days_shift)

    # clamp inside valid API range
    if candidate < MIN_DATE or candidate > MAX_DATE:
        return None, None

    return candidate, days_shift


def classify_shift(days_shift, shift_stats):
    if days_shift < 0:
        shift_stats["negative"] += 1
    else:
        shift_stats["positive"] += 1
        
    if abs(days_shift) <= 30:
        shift_stats["short"] += 1
    else:
        shift_stats["long"] += 1



def is_allowed(days_shift, shift_stats):
    # short vs long balance
    if abs(days_shift) <= 30:
        if shift_stats["short"] > shift_stats["long"] * 1.5:
            return False
    else:
        if shift_stats["long"] > shift_stats["short"] * 1.5:
            return False

    # direction balance
    if days_shift < 0:
        if shift_stats["negative"] > shift_stats["positive"] * 1.2:
            return False
    else:
        if shift_stats["positive"] > shift_stats["negative"] * 1.2:
            return False
    
    return True

    
def is_valid_candidate(location, candidate_date, fire_set):

    candidate_date = pd.to_datetime(candidate_date)

    candidate_year, candidate_week, _ = (
        candidate_date.isocalendar()
    )

    for fire_location, fire_date in fire_set:

        if fire_location != location:
            continue

        fire_date = pd.to_datetime(fire_date)

        fire_year, fire_week, _ = (
            fire_date.isocalendar()
        )

        if (
            fire_year == candidate_year
            and fire_week == candidate_week
        ):
            return False

    return True

In [ ]:
def make_key(forest, date, mode):
    return (forest, date, mode)

def allow_forest(original_forest, forest):

    forest_frequency = forest_context[forest_context['FORET'] == forest].iloc[0]['FORET_FIRE_COUNT']
    # attach to 0 if null
    forest_frequency = 0 if pd.isna(forest_frequency) else forest_frequency

    if pd.isna(forest_frequency):
        forest_frequency = 1
    max_allowed = foret_base_rep + forest_frequency * 0.2
    if(forest_usage[forest] >= max_allowed):
        return 0
    
    usage_penalty = 1 / (
        1 + forest_usage[forest]
    )

    pair_penalty = 1 / (
        1 + pair_usage[
            (original_forest, forest)
        ]
    )

    weight = (
        (forest_frequency + 1)
        * usage_penalty
        * pair_penalty
    )

    return weight

def calculate_similarity(row, candidate):
    
    score = 0

    if (
        row['FORET_GROUP']
        == candidate['FORET_GROUP']
    ):
        score += 2

    if (
        row['DAIRA_GROUP']
        == candidate['DAIRA_GROUP']
    ):
        score += 1

    if (
        row['COMMUNE_GROUP']
        == candidate['COMMUNE_GROUP']
    ):
        score += 1

    # -------------------------
    # 2. Vegetation similarity
    # -------------------------
    foret_diff = abs(
        row['TOT_FORET']
        - candidate['TOT_FORET']
    )

    maquis_diff = abs(
        row['TOT_MAQUIS']
        - candidate['TOT_MAQUIS']
    )

    broussailles_diff = abs(
        row['TOT_BROUSSAILLES']
        - candidate['TOT_BROUSSAILLES']
    )

    surf_diff = abs(
        row['SURF_TOTAL']
        - candidate['SURF_TOTAL']
    )

    # dense forest similarity
    if foret_diff < 10:
        score += 3
    elif foret_diff < 30:
        score += 1

    # maquis similarity
    if maquis_diff < 10:
        score += 3
    elif maquis_diff < 30:
        score += 1

    # broussailles similarity
    if broussailles_diff < 10:
        score += 3
    elif broussailles_diff < 30:
        score += 1

    if surf_diff < 10:
        score += 3
    elif surf_diff < 30:
        score += 1

    return score

In [ ]:
#candiante generation

def temporal_candidate(row, fire_set, shift_stats):
    
    forest = row['FORET']
    base_date = row['DATE']
    
    for _ in range(10):
        
        candidate_date, days_shift = random_shift_date(base_date)

        if candidate_date is None or days_shift is None:
            continue
        
        if not is_allowed(days_shift, shift_stats):
            continue
        
        if is_valid_candidate(forest, candidate_date, fire_set) is False:
            continue
        
        key = make_key(forest, candidate_date, "T")
        
        if key in seen_samples:
            continue
        
        seen_samples.add(key)
        return forest, candidate_date
    
    return None



In [ ]:
def cross_candidate(row, fire_set):

    date = row['DATE']
    original_forest = row['FORET']

    original_ctx = (
        forest_context[
            forest_context['FORET']
            == original_forest
        ]
        .iloc[0]
    )

    similar_forests = []
    similar_weights = []

    different_forests = []
    different_weights = []

    for forest in forest_pool:

        # skip same forest
        if forest == original_forest:
            continue

        # temporal validation
        if not is_valid_candidate(
            forest,
            date,
            fire_set
        ):
            continue

        key = make_key(
            forest,
            date,
            "C"
        )

        # already generated
        if key in seen_samples:
            continue

        # dynamic weight
        weight = allow_forest(
            original_forest,
            forest
        )

        # rejected by cap
        if weight <= 0:
            continue

        candidate_ctx = (
            forest_context[
                forest_context['FORET']
                == forest
            ]
            .iloc[0]
        )

        similarity = calculate_similarity(
            original_ctx,
            candidate_ctx
        )

        # --------------------
        # similar forests
        # --------------------

        if similarity >= 6:

            similar_forests.append(
                forest
            )

            similar_weights.append(
                weight
            )

        # --------------------
        # different forests
        # --------------------

        elif similarity <= 2:

            different_forests.append(
                forest
            )

            different_weights.append(
                weight
            )

    # --------------------
    # choose pool
    # --------------------

    use_similar = (
        random.random() < 0.7
    )

    if (
        use_similar
        and len(similar_forests) > 0
    ):

        selected_pool = similar_forests
        selected_weights = similar_weights

    elif len(different_forests) > 0:

        selected_pool = different_forests
        selected_weights = different_weights

    elif len(similar_forests) > 0:

        selected_pool = similar_forests
        selected_weights = similar_weights

    else:
        return None

    # weighted sampling
    new_forest = random.choices(
        selected_pool,
        weights=selected_weights,
        k=1
    )[0]

    # bookkeeping
    seen_samples.add(
        make_key(
            new_forest,
            date,
            "C"
        )
    )

    forest_usage[new_forest] += 1

    pair_usage[
        (original_forest, new_forest)
    ] += 1

    return new_forest, date

In [ ]:
def generate_label0_unified(df_fire, ratio=0.65):
    
    # =========================
    # INIT
    # =========================
    fire_set = set(zip(df_fire['FORET'], df_fire['DATE']))
    shift_stats = init_shift_stats()
    
    target_size = int(len(df_fire) * 2)
    temporal_target = int(target_size * ratio)
    
    label0 = []
    seen = set()
    
    i_temp = 0
    i_cross = 0
    
    attempts = 0
    max_attempts = target_size * 20
    
    rows = df_fire.sample(frac=1).reset_index(drop=True)
    
    # =========================
    # MAIN LOOP
    # =========================
    while len(label0) < target_size and attempts < max_attempts:
        
        row = rows.sample(1).iloc[0]
        attempts += 1

        # MODE SELECTION (STRICT 65/35)
        if i_temp < temporal_target:
            mode = "T"
        else:
            mode = "C"

        # GENERATE CANDIDATE
        if mode == "T":
            res = temporal_candidate(row, fire_set, shift_stats)
        else:
            res = cross_candidate(row, fire_set)
        
        if res is None:
            continue
        
        forest, date = res

        # DEDUPLICATION
        key = (forest, date, mode)
        if key in seen:
            continue
        seen.add(key)
        
        # BUILD FINAL ROW
        if mode == "T":
            new_row = change_row(row, date)
            i_temp += 1
        else:
            new_row = change_row_forest(row, forest)
            i_cross += 1
        
        new_row['LABEL'] = 0
        label0.append(new_row)
    
    return pd.DataFrame(label0)

In [ ]:
len(df15)

In [ ]:
weather_number = 0
df16 = df15.copy()
df_label0 = generate_label0_unified(df16)

In [ ]:
len(df_label0)

In [ ]:
df_label1 = df16.copy()
df_label1['LABEL'] = 1
df_label0 = df_label0[df_label1.columns]
df16 = pd.concat([df_label1, df_label0], axis=0).reset_index(drop=True)
df16 = df16.sample(frac=1).reset_index(drop=True)

In [ ]:
len(df16)

In [ ]:
#for train
#df16.to_excel("Cleaned Data/Theme1/split_train.xlsx", index=False)

In [ ]:
#for test
df16.to_excel("Cleaned Data/Theme1/split_test.xlsx", index=False)

## 👨🏻‍💻 Prepare for Model

In [ ]:
#for train
df17 = pd.read_excel("Cleaned Data/Theme1/split_train.xlsx")

In [ ]:
#for test
df17 = pd.read_excel("Cleaned Data/Theme1/split_test.xlsx")

In [ ]:
df17['COMMUNE'] = df17['COMMUNE_2']
df17['FORET'] = df17['FORET_2']
df17['DAIRA'] = df17['DAIRA_2']
df17 = df17.drop(columns=['COMMUNE_2', 'FORET_2', 'DAIRA_2', 'WEEK', 'DATE', 'ESSENCE', 
    'FORET_PERCENT', 'MAQUIS_PERCENT', 'BROUSSAILLES_PERCENT'])

In [ ]:
def encode_multilabel_fixed(df, col, values, prefix):
    

    split_series = df[col].fillna("").apply(lambda x: x.split("+"))

    for val in values:
        df[f"{prefix}_{val}"] = split_series.apply(lambda x: int(val in x))

    return df.drop(columns=[col])


def encode_onehot_fixed(df, col, values, prefix):

    for val in values:
        df[f"{prefix}_{val}"] = (df[col] == val).astype(int)

    return df.drop(columns=[col])


df18 = df17.copy()

# --- multi-label features ---
# df18 = encode_multilabel_fixed(df18, "ESSENCE", const_essences, "ESSENCE")
df18 = encode_multilabel_fixed(df18, "METEO_VENTE_DIRECTION", const_wind_directions, "WIND")

# --- single-label categorical ---
df18 = encode_onehot_fixed(df18, "SEASON", const_seasons, "SEASON")

cat_cols = ['DAIRA', 'COMMUNE', 'FORET', 'DAIRA_GROUP', 'FORET_GROUP', 'COMMUNE_GROUP', 'MONTH']

df18 = pd.get_dummies(df18, columns=cat_cols, drop_first=False)

df18 = df18.loc[:, ~df18.columns.str.endswith("_OTHER")]


#Test droping location columns
drop_cols = [
    col for col in df18.columns
    if (
        (
            col.startswith("FORET_")
            or col.startswith("COMMUNE_")
             or col.startswith("DAIRA_")
        )
        and ("_FIRE_COUNT" not in col)
        #and ("_PERCENT" not in col)
        and ("_GROUP" not in col)
    )
]
drop_cols += [
    col for col in df18.columns
    if (

        False

        
        )
]
#test removing essence
#drop_cols += [col for col in df18.columns if col.startswith("ESSENCE_")]

df18 = df18.drop(columns=drop_cols)

print("Final shape:", df18.shape)

In [ ]:
# for train
df18.to_excel("Cleaned Data/Theme1/train.xlsx", index=False)

NameError: name 'df18' is not defined

In [ ]:
# for test
df18.to_excel("Cleaned Data/Theme1/test.xlsx", index=False)

## 🤖 AI Model

In [ ]:
df_train = pd.read_excel("Cleaned Data/Theme1/train.xlsx")
df_train = df_train.dropna()
X_train = df_train.drop(columns=['LABEL'])
y_train = df_train['LABEL']


df_test = pd.read_excel("Cleaned Data/Theme1/test.xlsx")
df_test = df_test.dropna()
X_test = df_test.drop(columns=['LABEL'])
y_test = df_test['LABEL']

# ALIGN
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')

model.fit(X_train, y_train)
# predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

# print
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.7637
Precision: 0.6129
Recall   : 0.6667
F1-score : 0.6387
ROC-AUC  : 0.8284

Confusion Matrix:
[[101  24]
 [ 19  38]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.81      0.82       125
           1       0.61      0.67      0.64        57

    accuracy                           0.76       182
   macro avg       0.73      0.74      0.73       182
weighted avg       0.77      0.76      0.77       182



In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

# =========================
# 1. SCALE DATA (IMPORTANT)
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 2. MODEL
# =========================
model = SVC(
    kernel='rbf',       # nonlinear model
    C=1.0,              # regularization
    gamma='scale',      # auto kernel width
    probability=True,   # needed for ROC-AUC
    random_state=42
)

# =========================
# 3. TRAIN
# =========================
model.fit(X_train_scaled, y_train)

# =========================
# 4. PREDICTIONS
# =========================
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# =========================
# 5. METRICS
# =========================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

# =========================
# 6. RESULTS
# =========================
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.7088
Precision: 0.5500
Recall   : 0.3860
F1-score : 0.4536
ROC-AUC  : 0.7764

Confusion Matrix:
[[107  18]
 [ 35  22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.86      0.80       125
           1       0.55      0.39      0.45        57

    accuracy                           0.71       182
   macro avg       0.65      0.62      0.63       182
weighted avg       0.69      0.71      0.69       182



In [ ]:
# try random forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_model.fit(X_train, y_train)
# predictions
rf_y_pred = rf_model.predict(X_test)
rf_y_proba = rf_model.predict_proba(X_test)[:, 1]
# metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

# print
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy : 0.7088
Precision: 0.5500
Recall   : 0.3860
F1-score : 0.4536
ROC-AUC  : 0.7764

Confusion Matrix:
[[107  18]
 [ 35  22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.86      0.80       125
           1       0.55      0.39      0.45        57

    accuracy                           0.71       182
   macro avg       0.65      0.62      0.63       182
weighted avg       0.69      0.71      0.69       182



In [ ]:
#merge forest and xgboost

proba1 = model.predict_proba(X_test)[:, 1]
proba2 = rf_model.predict_proba(X_test)[:, 1]

# ==========================================
# SOFT VOTING
# ==========================================

final_proba = (proba1 + proba2) / 2

# threshold
threshold = 0.5

y_pred = (final_proba >= threshold).astype(int)
# metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, final_proba)
# print
print(f"Ensemble Accuracy : {accuracy:.4f}")
print(f"Ensemble Precision: {precision:.4f}")
print(f"Ensemble Recall   : {recall:.4f}")
print(f"Ensemble F1-score : {f1:.4f}")
print(f"Ensemble ROC-AUC  : {roc_auc:.4f}")

Ensemble Accuracy : 0.7033
Ensemble Precision: 1.0000
Ensemble Recall   : 0.0526
Ensemble F1-score : 0.1000
Ensemble ROC-AUC  : 0.8229


d:\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


In [ ]:
# use cat boost
cb_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bylevel=0.8,
    eval_metric='Logloss',
    random_seed=42,
    verbose=False
)
cb_model.fit(X_train, y_train)
# predictions
y_pred = cb_model.predict(X_test)
y_proba = cb_model.predict_proba(X_test)[:, 1]
# metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

# print
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.7034
Precision: 0.6500
Recall   : 0.2385
F1-score : 0.3490
ROC-AUC  : 0.8485

Confusion Matrix:
[[204  14]
 [ 83  26]]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.94      0.81       218
           1       0.65      0.24      0.35       109

    accuracy                           0.70       327
   macro avg       0.68      0.59      0.58       327
weighted avg       0.69      0.70      0.65       327



In [ ]:
# LSTM

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Input

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_lstm = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    1,
    X_train_scaled.shape[1]
)

X_test_lstm = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    1,
    X_test_scaled.shape[1]
)

model = Sequential([
    Input(shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),

    LSTM(64),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_lstm,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

### 💾 Save model and assets

In [ ]:
real_forest_context = forest_context.drop(columns = ['COMMUNE_2', 'FORET_2', 'DAIRA_2', 'ESSENCE', 
    'FORET_PERCENT', 'MAQUIS_PERCENT', 'BROUSSAILLES_PERCENT'])
real_forest_context.columns

In [ ]:
base_path = "Cleaned Data/Theme1/assets"

joblib.dump(
    model,
    f"{base_path}/model.pkl"
)
joblib.dump(
    X_train.columns.tolist(),
    f"{base_path}/features.pkl"
)
joblib.dump(
    real_forest_context,
    f"{base_path}/forest_context.pkl"
)